# KC-locked Morlet-wavelet spectrogram (Figure 2C)

Self-contained: loads the cleaned data, builds KC-locked epochs, computes a
**Morlet wavelet** time-frequency representation with
`mne.time_frequency.tfr_array_morlet`, and plots the grand-average (or a single subject).

Morlet (n_cycles = freqs/2) avoids both the STFT windowing ringing and the DPSS
multitaper-taper interference against the KC's sharp edge. Epochs are cut wide
(±6 s) so wavelet edge effects fall outside the ±3 s display window. Tweak the
**PARAMS** cell to change the look.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from mne.time_frequency import tfr_array_morlet
mne.set_log_level('ERROR')

# ---- paths -----------------------------------------------------------------
# cleaned data at <DATA_DIR>/<subject>/cleaned/<subject>_cleaned_raw.fif (+ _annotations.csv)
DATA_DIR = Path(os.environ.get('KC_DATA_DIR', '~/Desktop/SS2_Results/New')).expanduser()
SUBJECTS = [f'01-02-{i:04d}' for i in range(1, 20)]   # all 19
FS = 256
print(DATA_DIR, '|', len(SUBJECTS), 'subjects')

In [ ]:
# ---- data loading (same as the analysis package) ---------------------------
def _kc_spindle_onsets(desc, onset):
    grp = desc.str.extract(r'[gG]roup[nN]ame="([^"]+)"')[0].fillna(desc).str.lower()
    kc = onset[grp.str.contains('kcomplex').to_numpy()]
    sp = onset[grp.str.contains('spindle').to_numpy()]
    return kc.astype(float), sp.astype(float)

def load_subject(sid, data_root=DATA_DIR):
    d = Path(data_root) / sid / 'cleaned'
    raw = mne.io.read_raw_fif(d / f'{sid}_cleaned_raw.fif', preload=False)
    c3 = next(c for c in raw.ch_names if 'C3' in c.upper())
    sig = raw.get_data(picks=[c3], units='uV')[0].astype(float)
    fs = int(round(raw.info['sfreq']))
    ann = pd.read_csv(d / f'{sid}_annotations.csv')
    kc, sp = _kc_spindle_onsets(ann['description'].astype(str),
                                ann['onset_s'].astype(float).to_numpy())
    return dict(sig=sig, fs=fs, kc_onsets=kc, spindle_onsets=sp)

def align_to_negpeak(sig, fs, onsets, search_s=1.0):
    peaks = []
    for on in onsets:
        c = int(round(on * fs)); a, b = c, min(c + int(search_s * fs), len(sig))
        if b - a > 10:
            peaks.append(a + int(np.argmin(sig[a:b])))
    return np.array(peaks, int)

In [ ]:
# ============================ PARAMS (tweak me) =============================
HALF        = 6.0          # epoch half-width (s); wider than the display so the
                           # wavelet cone-of-influence falls outside the plot
DISPLAY     = 3.0          # +/- seconds actually shown
FMIN, FMAX  = 1.0, 30.0    # frequency range (Hz)
FSTEP       = 0.5          # frequency spacing (Hz)
N_CYCLES    = 'freqs/2'    # 'freqs/2' (adaptive) OR a number like 5 (fixed)
DECIM       = 4            # temporal decimation of the TFR
MAX_EVENTS  = 120          # KCs per subject
BASELINE    = 2.0          # subtract per-freq mean over |t| > BASELINE (s)
# ===========================================================================

freqs = np.arange(FMIN, FMAX + 1e-9, FSTEP)
n_cycles = (freqs / 2.0) if N_CYCLES == 'freqs/2' else float(N_CYCLES)
print(f'{len(freqs)} freqs {FMIN}-{FMAX} Hz | n_cycles={N_CYCLES} | half={HALF}s')

In [ ]:
# ---- compute the mean Morlet TFR for one subject ---------------------------
def subject_tfr(sid):
    d = load_subject(sid); sig, fs = d['sig'], int(round(d['fs']))
    peaks = align_to_negpeak(sig, fs, d['kc_onsets'])[:MAX_EVENTS]
    ep = []
    for c0 in peaks:
        c0 = int(c0); a, b = c0 - int(HALF*fs), c0 + int(HALF*fs)
        if a < 0 or b > len(sig):
            continue
        ep.append(sig[a:b])
    X = np.asarray(ep)[:, None, :]                    # (n_epochs, 1, n_times)
    power = tfr_array_morlet(X, sfreq=fs, freqs=freqs, n_cycles=n_cycles,
                             output='power', zero_mean=True, decim=DECIM, verbose=False)
    meanLogS = np.log10(power.mean(0)[0] + 1e-30)     # (freq, time)
    times = np.linspace(-HALF, HALF, X.shape[-1])[::DECIM][:meanLogS.shape[1]]
    wave = np.asarray(ep).mean(0)
    twave = np.linspace(-HALF, HALF, len(wave))
    return meanLogS, times, wave, twave, len(ep)

# quick single-subject test
mLS, times, wave, twave, n = subject_tfr(SUBJECTS[0])
print('one subject ok:', mLS.shape, 'n=', n)

In [ ]:
# ---- loop all subjects and grand-average -----------------------------------
acc, waves, times = None, [], None
for sid in SUBJECTS:
    try:
        mLS, times, wave, twave, n = subject_tfr(sid)
    except Exception as e:
        print('skip', sid, e); continue
    acc = mLS if acc is None else acc + mLS
    waves.append(wave)
    print(f'{sid}: n={n}')
S = acc / len(waves)                                  # grand-mean log power (freq,time)
wave = np.mean(waves, 0)
print('grand average over', len(waves), 'subjects')

In [ ]:
# ---- plot -------------------------------------------------------------------
Sb = S - S[:, np.abs(times) > BASELINE].mean(1, keepdims=True)   # baseline subtract

fig, ax = plt.subplots(figsize=(7.2, 4.6))
pcm = ax.pcolormesh(times, freqs, Sb, cmap='magma', shading='gouraud')
cb = plt.colorbar(pcm, ax=ax, fraction=.046, pad=.02); cb.set_label('\u0394 log\u2081\u2080 power')
wv = wave - wave.mean(); wv = 7 + (wv / np.max(np.abs(wv))) * 7
ax.plot(twave, wv, color='w', lw=1.8, path_effects=[pe.withStroke(linewidth=3, foreground='k')])
ax.set_xlim(-DISPLAY, DISPLAY); ax.set_ylim(FMIN, FMAX)
ax.set_xlabel('time relative to K-complex (s)'); ax.set_ylabel('frequency (Hz)')
plt.tight_layout(); plt.show()